# Plant disease detection with MobileNetV2

This Colab-ready notebook expects this folder layout:
```text
data/train/<class_name>/*.jpg
data/val/<class_name>/*.jpg
data/test/<class_name>/*.jpg
```

Upload or mount the folder before running the training cells.

In [ ]:
# Install the minimal Colab dependencies.
!pip -q install tensorflow matplotlib

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = 'data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
TEST_DIR = os.path.join(DATA_DIR, 'test')

if not os.path.isdir(TRAIN_DIR):
    raise FileNotFoundError('Upload a data/train folder before running this notebook.')

In [ ]:
# Load each split from folders. The validation and test sets reuse the
# training class order so labels line up with the final model output.
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='int', shuffle=True, seed=SEED
)
class_names = train_ds.class_names
num_classes = len(class_names)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='int', class_names=class_names, shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='int', class_names=class_names, shuffle=False
)
print('Classes:', class_names)
print('Number of classes:', num_classes)

In [ ]:
# Prefetch improves input throughput without changing the dataset.
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
], name='data_augmentation')

In [ ]:
# Transfer-learning model.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet'
)
# To try EfficientNet-B0 instead, use:
# base_model = tf.keras.applications.EfficientNetB0(
#     input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet'
# )
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Phase 1: train only the new classification head for 5 epochs.
history_frozen = model.fit(train_ds, validation_data=val_ds, epochs=5)

In [ ]:
# Phase 2: fine-tune only the top part of the backbone for 5 more epochs.
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
# Keep BatchNorm stable during small hackathon datasets.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_finetuned = model.fit(train_ds, validation_data=val_ds, epochs=5)

In [ ]:
# Evaluate and save the exact model used by the FastAPI endpoint.
test_loss, test_accuracy = model.evaluate(test_ds)
print(f'Test accuracy: {test_accuracy:.3f}')
model.save('plant_disease_mobilenet.h5')
print('Saved plant_disease_mobilenet.h5')

# Keep this list in the same order as class_names when configuring the API.
print('API class names:', class_names)